In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    LogisticRegression
)

from sklearn.metrics import (
    mean_squared_error,
    r2_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from imblearn.over_sampling import SMOTE

In [ ]:
df = pd.read_csv("data/cleaned_data.csv")

print(df.head())

In [ ]:
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns)

print("\nData Types:")
print(df.dtypes)

In [ ]:
# Defining regression and classification targets

X = df.drop(columns=["avg_glucose_level", "stroke"])

y_reg = df["avg_glucose_level"]

y_clf = df["stroke"]

print("Features shape:", X.shape)
print("Regression target:", y_reg.shape)
print("Classification target:", y_clf.shape)

In [ ]:
# One hot encoding categorical columns

X_encoded = pd.get_dummies(
    X,
    drop_first=True
)

print(X_encoded.head())
print(X_encoded.shape)

In [ ]:
X_train, X_test, y_reg_train, y_reg_test = train_test_split(
    X_encoded,
    y_reg,
    test_size=0.2,
    random_state=42
)

X_train_clf, X_test_clf, y_clf_train, y_clf_test = train_test_split(
    X_encoded,
    y_clf,
    test_size=0.2,
    random_state=42
)


scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_clf_scaled = scaler.fit_transform(X_train_clf)
X_test_clf_scaled = scaler.transform(X_test_clf)


print("Scaling completed")

In [ ]:
lr = LinearRegression()

lr.fit(
    X_train_scaled,
    y_reg_train
)

y_pred_reg = lr.predict(X_test_scaled)


mse = mean_squared_error(
    y_reg_test,
    y_pred_reg
)

r2 = r2_score(
    y_reg_test,
    y_pred_reg
)


print("Linear Regression MSE:", mse)
print("Linear Regression R2:", r2)

In [ ]:
coef_df = pd.DataFrame({
    "Feature": X_encoded.columns,
    "Coefficient": lr.coef_
})

coef_df["Absolute"] = abs(coef_df["Coefficient"])

print(
    coef_df.sort_values(
        "Absolute",
        ascending=False
    ).head(3)
)

In [ ]:
ridge = Ridge(alpha=1.0)

ridge.fit(
    X_train_scaled,
    y_reg_train
)

ridge_pred = ridge.predict(X_test_scaled)


ridge_mse = mean_squared_error(
    y_reg_test,
    ridge_pred
)

ridge_r2 = r2_score(
    y_reg_test,
    ridge_pred
)


comparison = pd.DataFrame({
    "Model":["Linear Regression","Ridge Regression"],
    "MSE":[mse,ridge_mse],
    "R2":[r2,ridge_r2]
})


comparison

In [ ]:
print(y_clf_train.value_counts())

In [ ]:
log_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)


log_model.fit(
    X_train_clf_scaled,
    y_clf_train
)


y_pred = log_model.predict(
    X_test_clf_scaled
)


y_prob = log_model.predict_proba(
    X_test_clf_scaled
)[:,1]


print(confusion_matrix(
    y_clf_test,
    y_pred
))

print(
    classification_report(
        y_clf_test,
        y_pred
    )
)

In [ ]:
fpr, tpr, thresholds = roc_curve(
    y_clf_test,
    y_prob
)

auc = roc_auc_score(
    y_clf_test,
    y_prob
)


plt.figure(figsize=(6,4))

plt.plot(
    fpr,
    tpr,
    label=f"AUC={auc:.3f}"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title("ROC Curve")

plt.legend()

plt.savefig(
    "plots/roc_curve.png"
)

plt.show()

In [ ]:
results=[]

for threshold in np.arange(0.3,0.8,0.1):

    pred = (
        y_prob >= threshold
    ).astype(int)

    results.append([
        threshold,
        precision_score(y_clf_test,pred),
        recall_score(y_clf_test,pred),
        f1_score(y_clf_test,pred)
    ])


threshold_table=pd.DataFrame(
    results,
    columns=[
        "Threshold",
        "Precision",
        "Recall",
        "F1"
    ]
)

threshold_table

In [ ]:
log_c001 = LogisticRegression(
    C=0.01,
    max_iter=1000,
    class_weight="balanced"
)

log_c001.fit(
    X_train_clf_scaled,
    y_clf_train
)


prob_c001 = log_c001.predict_proba(
    X_test_clf_scaled
)[:,1]


auc_c001 = roc_auc_score(
    y_clf_test,
    prob_c001
)


print("Baseline AUC:", auc)
print("C=0.01 AUC:", auc_c001)

In [ ]:
print("PART 2 COMPLETED")